# Table 6.2 — hedging PE, TI and EI portfolios

This notebook reproduces the experimental design and table layout using the current
CarbonFund_UL engine: three portfolios, common fund/carbon scenarios, and three
hedging strategies per portfolio. Every simulation input is specified below.

EI pays the ordinary PE survival benefit or a fraction of the TI death benefit;
payments are discounted once. All reported costs use the corrected **time-zero**
accounting, with positive costs meaning additional funding. The historical paper
numbers are retained separately: the legacy accounting corrections mean numerical
equality with those numbers is not an acceptance test.

Select the project's `.venv-analysis` kernel in VS Code. Set the paths and execution
switches below. `RUN_SIMULATIONS=False` reloads completed runs for analysis;
`OVERWRITE_EXISTING=True` explicitly permits replacing generated results in this
experiment directory. No changes to the C++ library are required.


In [ ]:
from pathlib import Path
from copy import deepcopy
import csv
import html
import json
import sys

SEARCH_START = Path.cwd().resolve()
PROJECT_ROOT = next(
    (parent for parent in (SEARCH_START, *SEARCH_START.parents)
     if (parent / 'include' / 'carbonfund').is_dir()
     and (parent / 'python' / 'carbonfund_experiments.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start the notebook from the CarbonFund_UL folder or a subdirectory.')
BINARY = PROJECT_ROOT / 'build-paper-release' / ('carbonfund_experiment.exe' if sys.platform == 'win32' else 'carbonfund_experiment')
OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'table_6_2'
RUN_NAME = 'cir_T20_min16_01'
RUN_SIMULATIONS = True
OVERWRITE_EXISTING = False
SAVE_FIGURES = True
SHOW_FIGURES = True

sys.path.insert(0, str(PROJECT_ROOT / 'python'))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import carbonfund_experiments as cfe

RUN_DIRECTORY = OUTPUT_ROOT / RUN_NAME
if RUN_SIMULATIONS and not BINARY.is_file():
    raise FileNotFoundError(f'Build carbonfund_experiment first: {BINARY}')
print('Python module:', Path(cfe.__file__).resolve())
print('Results:', RUN_DIRECTORY)


**Market inputs.** The dimension is fixed to four stocks. Returns, volatilities
and the continuously compounded risk-free rate are annual quantities. Covariance
is derived from volatilities and correlations. The matrix below uses the full
precision of the legacy reference code; Table 4.1 prints rounded entries.


In [ ]:
market = {
    'risk_free_rate': 0.05,
    'expected_returns': [0.25, 0.15, 0.10, 0.08],
    'volatilities': [0.30, 0.25, 0.20, 0.16],
    'correlations': [
        [1.0000, 0.4397, 0.3900, 0.3168],
        [0.4397, 1.0000, 0.2954, 0.3261],
        [0.3900, 0.2954, 1.0000, 0.3134],
        [0.3168, 0.3261, 0.3134, 1.0000],
    ],
}


**Carbon inputs.** For the paper experiment use independent CIR intensities,
$dC_i=\kappa_i(\beta_i-C_i)dt+s_i\sqrt{C_i}\,dW_i$, simulated with the CIR
Ninomiya–Victoir stepper. `initial`, `long_run_mean`, `mean_reversion` and
`volatility` each contain four entries. The implementation requires
$\kappa_i\beta_i\ge s_i^2/4$. Carbon and financial innovations are independent.

The interface also accepts `OU` with the same field names, or `GeoOU` with
`long_run_log_mean` replacing `long_run_mean`. Those are different experiments.


In [ ]:
carbon = {
    'model': 'CIR',
    'initial': [5000.0, 4000.0, 3000.0, 1000.0],
    'long_run_mean': [2500.0, 2000.0, 1500.0, 500.0],
    'mean_reversion': [0.05, 0.05, 0.05, 0.05],
    'volatility': [3.0, 3.0, 3.0, 3.0],
}


**Fund preferences.** `risk_aversion` is $\delta>0$ and `carbon_penalty` is
$\alpha\ge0$. These inputs determine the four stock allocations and consequently
the reference fund's dynamics. All three insurance portfolios use this same fund.


In [ ]:
fund = {
    'risk_aversion': 1.0,
    'carbon_penalty': 0.0025,
}


**Contract inputs.** The fund starts at `fund_value`, maturity is in years, and
guarantees remain anchored to the original contract:
$L(t)=L_0e^{g_Lt}$ and $U(t)=U_0e^{g_Ut}$. The benefit per unit of notional is
$g(t,x)=\min(U(t),\max(L(t),x))$.

PE pays $g(T,X_T)$ on survival; TI pays $g(\tau,X_\tau)$ on death before maturity;
EI pays the PE benefit on survival or $\rho g(\tau,X_\tau)$ on death.
`endowment_death_benefit_fraction` is $\rho$. Ordinary discounting is applied by
the engine. Initial guarantees may coincide when future guarantees separate;
every future pricing date must have a strictly smaller floor than cap.


In [ ]:
contract = {
    'fund_value': 1.0,
    'maturity': 20.0,
    'minimum_guarantee': 1.0,
    'maximum_guarantee': 1.0,
    'minimum_guarantee_growth_rate': 0.05,
    'maximum_guarantee_growth_rate': 0.50,
    'endowment_death_benefit_fraction': 1.0,
}


**Mortality inputs.** The Gompertz–Makeham intensity for initial age $a$ is
$$\lambda_a(t)=\xi+\frac{1}{b}\exp\!\left(\frac{a+t-m}{b}\right).$$
The JSON field `b` is the scale, not its reciprocal. Use the corrected
$\xi=0.0041959$. The same mortality law is used in all three portfolios.


In [ ]:
mortality = {
    'xi': 0.0041959,
    'm': 79.6921211,
    'b': 11.5818911,
}


**Population inputs.** Table 6.2 uses 1,000 policyholders aged 60 in each portfolio.
The legacy reference code sets **total portfolio notional to one**, so the default
individual notional below is 0.001. Setting each notional to one would multiply
aggregate means and standard deviations by 1,000.

Edit the count, age or notional independently for each portfolio. For a heterogeneous
population, replace a generated list with explicit `type`, `age`, `notional` records.
To represent the same lives across PE/TI/EI, preserve their ordering and ages.


In [ ]:
population_inputs = {
    'PE': {'count': 1000, 'age': 60.0, 'notional': 0.001},
    'TI': {'count': 1000, 'age': 60.0, 'notional': 0.001},
    'EI': {'count': 1000, 'age': 60.0, 'notional': 0.001},
}

populations = {
    kind: [
        {'type': kind, 'age': p['age'], 'notional': p['notional']}
        for _ in range(p['count'])
    ]
    for kind, p in population_inputs.items()
}
for kind, people in populations.items():
    print(kind, 'policies:', len(people),
          'total notional:', sum(p['notional'] for p in people))


**Outer simulation inputs.** `outer_scenarios` counts independent physical-measure
market/mortality scenarios. `intervals` determines the dates of market observation,
death settlement and hedge rebalancing. `threads` parallelizes these scenarios:
1 is serial, 0 uses the OpenMP default, and a positive value requests that team size.
`progress_every=100` prints after every 100 completed scenarios; zero disables it.

The three seeds below belong to distinct carbon, financial and mortality streams.
Reuse the same values in all three portfolios. The initial/revaluation seeds in
the next cells must also remain distinct within each run. Keeping the entire seed
set fixed reproduces the common market scenarios without storing a scenario cube.

The numerical budgets are editable choices, not a claim that they reproduce every
historical run setting. These full settings can take a substantial time to run.


In [ ]:
outer_simulation = {
    'outer_scenarios': 5000,
    'intervals': 1040,
    'threads': 16,
    'progress_every': 200,
    'carbon_seed': 2026101801,
    'financial_seed': 2026101802,
    'mortality_seed': 2026101803,
}


**Initial valuation inputs.** Each portfolio's premium and initial deltas are
computed once using its own liabilities. A more accurate initial valuation reduces
the uncertainty shared by every outer scenario. `paths` counts actual trajectories:
10,000 antithetic paths correspond to 5,000 independent pair averages.

Available configurations are `standard`, `standard_antithetic`, `conditioned` and
`conditioned_antithetic`. Conditioning integrates out financial risk in the pricing
calculation. Antithetics negate each four-dimensional Gaussian input and support CIR.


In [ ]:
initial_valuation = {
    'paths': 4000,
    'intervals': 1040,
    'configuration': 'conditioned_antithetic',
    'carbon_seed': 2026101804,
    'financial_seed': 2026101805,
}


**Revaluation inputs.** These settings price the surviving liabilities and compute
their deltas at every intermediate hedge date. `intervals=256` is the full-horizon
pricing budget. The residual grid uses
$N(t)=\max(N_{\min},\lceil256(T-t)/T\rceil)$, where `minimum_intervals=16`
sets $N_{\min}$. These counts refer to intervals; there are $N(t)+1$ grid points.
The minimum must be an integer between 1 and `intervals`. Omitting it preserves
the previous minimum of 1. Setting it equal to `intervals` fixes the interval
count throughout revaluation. It changes neither the outer hedge dates nor the
initial valuation grid. `paths=40` means 20 pairs for this configuration.

The floor matters when the remaining time is less than approximately
$T N_{\min}/256$. For $T=20$, a floor of 16 refines the final 1.25 years.
A larger minimum refines carbon integration and the conditional mortality/payment
grid; it does not add outer hedge dates or simulate the fund on the inner grid
for the conditioned estimator.

For a convergence comparison, use minima 1, 8, 16 and 32 in separate run directories,
keeping every other input and all seeds unchanged. The initial premium and outer
market/mortality outcomes are then identical. Inner grids change, so identical
inner seeds do not mean exactly coupled inner Brownian paths. Compare the paired
scenario cost differences as well as each mean and its interval. A confidence
interval excluding zero alone does not identify which discretization causes it;
it is conditional on the single computed initial premium.


In [ ]:
revaluation = {
    'paths': 40,
    'intervals': 256,
    'minimum_intervals': 16,
    'configuration': 'conditioned_antithetic',
    'carbon_seed': 2026101806,
    'financial_seed': 2026101807,
}


**Analysis inputs.** `aggregate` reports the whole portfolio, `per_policy` divides
costs by the number of policies, and `per_notional` divides by total notional.
The Table 6.2 layout contains means and sample standard deviations; the complete
summaries also contain variances, confidence intervals, quantiles and upper-tail ES.
The standard deviation uses `ddof=1`.

`comparison_family_size=9` covers the three strategy contrasts in each of three
portfolios. Use zero for correction separately within each portfolio. Plot ranges
and zoom are `[minimum, maximum]` or `None`. The confidence intervals for means
describe outer uncertainty conditional on the computed initial premium.


In [ ]:
analysis = {
    'normalization': 'aggregate',
    'confidence_level': 0.95,
    'comparison_family_size': 9,
    'quantiles': [0.025, 0.50, 0.90, 0.95, 0.975, 0.99],
    'expected_shortfall_levels': [0.95, 0.99],
    'bins': 70,
    'zoom': None,
    'plot_range': None,
    'figure_formats': ['png', 'pdf'],
    'dpi': 180,
    'title': 'Table 6.2: corrected time-zero hedging costs',
}
DISPLAY_DECIMALS = 4
NOTES = (
    'Shared market and mortality streams across PE/TI/EI. '
    'Ordinary EI benefits; one discount to time zero. '
    'Corrected xi and age+t-m mortality convention. '
    'Corrected discounted hedging accounts and static-delta scaling. '
    'Historical paper numbers use legacy accounting; exact equality is not required.'
)


**Assemble and inspect the configurations.** Every portfolio gets a deep copy of
the same common inputs and seed set, with its own population. No demo or paper
preset is loaded. Re-run this cell after changing an input cell: configurations
are snapshots, so edits to the original dictionaries do not update old copies.


In [ ]:
common_config = {
    'schema_version': 1,
    'experiment': 'hedging',
    'market': deepcopy(market),
    'carbon': deepcopy(carbon),
    'fund': deepcopy(fund),
    'contract': deepcopy(contract),
    'mortality': deepcopy(mortality),
    'population': [],
    'hedging': {
        **deepcopy(outer_simulation),
        'initial': deepcopy(initial_valuation),
        'revaluation': deepcopy(revaluation),
    },
    'analysis': deepcopy(analysis),
    'notes': NOTES,
}
configs = {}
for kind, people in populations.items():
    cfg = deepcopy(common_config)
    cfg['population'] = deepcopy(people)
    cfe.validate_config(cfg)
    configs[kind] = cfg

print(json.dumps({k: v for k, v in common_config.items() if k != 'population'}, indent=2))
for kind, cfg in configs.items():
    print(kind, 'policies:', len(cfg['population']),
          'total notional:', sum(p['notional'] for p in cfg['population']))


**Run or reload the three experiments.** Each run uses all three hedging strategies
on common outcomes. Outputs are saved in separate `PE`, `TI`, `EI` directories,
including raw `hedging.csv`, exact `config.json`, `metadata.json` and statistical
summaries. Raw CSV amounts are aggregate amounts; normalization is an analysis choice.

To analyse completed experiments later, set `RUN_SIMULATIONS=False`. The code checks
that simulation inputs agree with the saved inputs; analysis settings may change.
The actual OpenMP team size and the initial/outer wall times are printed below.


In [ ]:
def simulation_inputs(cfg):
    inputs = deepcopy({key: value for key, value in cfg.items()
                       if key not in ('analysis', 'notes')})
    inputs['hedging']['revaluation'].setdefault('minimum_intervals', 1)
    return inputs

results = {}
for kind, cfg in configs.items():
    destination = RUN_DIRECTORY / kind
    print(f'\n{kind} -> {destination}', flush=True)
    if RUN_SIMULATIONS:
        result = cfe.run_experiment(
            cfg, destination, binary=BINARY,
            overwrite=OVERWRITE_EXISTING, plots=SAVE_FIGURES, progress=True,
        )
    else:
        result = cfe.load_result(destination)
        if simulation_inputs(result.config) != simulation_inputs(cfg):
            raise ValueError(f'{kind}: current simulation inputs differ from the saved run.')
        result.config['analysis'] = deepcopy(analysis)
        result.save_analysis(plots=SAVE_FIGURES)
    results[kind] = result
    run = result.metadata['run']
    print('Initial price:', run['initial_price'])
    print('Initial PE/TI/EI deltas:', run['initial_deltas_PE_TI_EI'])
    print('Revaluation minimum intervals:', run.get('revaluation_minimum_intervals', 1))
    print('Actual threads:', run['threads_used'])
    print('Initial elapsed seconds:', run['initial_wall_seconds'])
    print('Outer elapsed seconds:', run['simulation_wall_seconds'])


**Check the common scenarios.** Scenario IDs and terminal fund values must agree
exactly across the three runs. The stream construction makes complete market paths
identical when the common inputs are identical; terminal values provide a saved-data
consistency check.

When the three lists represent the same lives with the same notionals and the
valuation settings are common, EI cash flows and each strategy's costs satisfy
$C_{EI}=C_{PE}+\rho C_{TI}$, up to floating-point roundoff. This also checks the
shared mortality outcomes and ordinary EI benefit convention.


In [ ]:
def read_raw(result):
    with (result.directory / 'hedging.csv').open(newline='') as stream:
        rows = list(csv.DictReader(stream))
    return {key: np.array([float(row[key]) for row in rows]) for key in rows[0]}

raw = {kind: read_raw(result) for kind, result in results.items()}
for kind in ('TI', 'EI'):
    np.testing.assert_array_equal(raw['PE']['scenario_id'], raw[kind]['scenario_id'])
    np.testing.assert_array_equal(raw['PE']['fund_terminal'], raw[kind]['fund_terminal'])
print('Common scenario IDs and terminal fund values: verified.')

lives = {kind: [(p['age'], p['notional']) for p in cfg['population']]
         for kind, cfg in configs.items()}
if lives['PE'] == lives['TI'] == lives['EI']:
    rho = contract['endowment_death_benefit_fraction']
    for key in ('claims_pv', 'initial_price', 'no_hedge', 'static_hedge', 'dynamic_hedge'):
        reference = raw['PE'][key] + rho * raw['TI'][key]
        scale = max(1.0, float(np.max(np.abs(reference))))
        np.testing.assert_allclose(raw['EI'][key], reference, rtol=1e-10, atol=1e-12 * scale)
        print(key, 'maximum EI - PE - rho*TI residual:',
              float(np.max(np.abs(raw['EI'][key] - reference))))
else:
    print('Cash-flow additivity check skipped: the three insured populations differ.')


**Create the Table 6.2 layout.** Each panel reports mean and sample standard
deviation for no hedging, static hedging and dynamic hedging. The paper calls the
last column continuous hedging; here it is implemented on the chosen discrete grid.
Display uses scientific notation with four decimals, while CSV export preserves
the numerical values. Normalization is the selected analysis normalization.


In [ ]:
strategy_order = ('no_hedge', 'static_hedge', 'dynamic_hedge')
strategy_labels = ('No hedging', 'Static hedging', 'Dynamic hedging')
contract_labels = {'PE': 'Pure Endowment', 'TI': 'Term Insurance', 'EI': 'Endowment Insurance'}
summaries = {kind: result.summary() for kind, result in results.items()}
table_rows = []
for kind in ('PE', 'TI', 'EI'):
    by_strategy = {row['strategy']: row for row in summaries[kind]}
    for statistic, key in (('Mean', 'mean'), ('St. Dev.', 'std')):
        table_rows.append({
            'contract': kind, 'statistic': statistic,
            **{strategy: by_strategy[strategy][key] for strategy in strategy_order},
        })

def render_table(rows):
    pieces = ['<table><thead><tr><th>Statistic</th>']
    pieces += [f'<th>{html.escape(label)}</th>' for label in strategy_labels]
    pieces.append('</tr></thead><tbody>')
    for kind in ('PE', 'TI', 'EI'):
        pieces.append(f'<tr><th colspan="4">{contract_labels[kind]}</th></tr>')
        for row in rows:
            if row['contract'] == kind:
                pieces.append('<tr><th>' + html.escape(row['statistic']) + '</th>')
                pieces.extend(f'<td>{row[strategy]:.{DISPLAY_DECIMALS}e}</td>' for strategy in strategy_order)
                pieces.append('</tr>')
    pieces.append('</tbody></table>')
    return ''.join(pieces)

print('Normalization:', analysis['normalization'], '| costs valued at time zero')
display(HTML(render_table(table_rows)))
RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
table_path = RUN_DIRECTORY / 'table_6_2.csv'
with table_path.open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(table_rows[0]))
    writer.writeheader()
    writer.writerows(table_rows)
print('Saved:', table_path)


**Historical paper values.** These are the numbers printed in Table 6.2, retained
for documentation. They are not an equality target for the corrected time-zero
hedging engine: the old dynamic costs mixed dates and the old static update
reapplied aggregate notionals. Matching scenarios alone does not reconcile those
accounting differences. Update the paper table from the corrected experiment
after checking convergence and the chosen normalization.


In [ ]:
paper_values = {
    'PE': {'Mean': [5.842, 2.186, -0.0016], 'St. Dev.': [2.840, 1.807, 0.1184]},
    'TI': {'Mean': [1.863, 0.503, -0.0004], 'St. Dev.': [0.184, 0.474, 0.1210]},
    'EI': {'Mean': [7.721, 2.708,  0.0006], 'St. Dev.': [2.846, 1.727, 0.0120]},
}
historical_rows = [
    {'contract': kind, 'statistic': statistic, **dict(zip(strategy_order, values))}
    for kind, statistics in paper_values.items()
    for statistic, values in statistics.items()
]
display(HTML(render_table(historical_rows)))


**Complete statistics.** The saved summaries include variance, standard deviation,
standard error and confidence interval for the mean, quantiles, ES and probability
of positive cost. The paired comparisons concern strategies applied to the same
outcomes within each portfolio. A significant mean conditional on one initial
premium can reflect initial pricing error; assess it with independent valuation
seeds and grid refinement as well as more outer scenarios.


In [ ]:
for kind, result in results.items():
    display(HTML(f'<h4>{contract_labels[kind]}: summary</h4>'))
    display(result.summary())
    display(HTML(f'<h4>{contract_labels[kind]}: paired strategy comparisons</h4>'))
    display(result.comparisons())


**Figures and later analysis.** Each figure contains densities with a central zoom,
empirical CDFs and box plots for the three strategies. When `SAVE_FIGURES=True`,
the selected formats are saved in each portfolio directory. You can change
analysis inputs and rerun the configuration, reload, table and figure cells with
`RUN_SIMULATIONS=False`; the Monte Carlo calculations are then reused.


In [ ]:
if SHOW_FIGURES:
    for kind, result in results.items():
        fig = result.plot()
        fig.suptitle(f'{contract_labels[kind]} — {analysis["normalization"]}, time-zero costs')
        display(fig)
        plt.close(fig)
else:
    print('Inline figures disabled by SHOW_FIGURES.')
